In [2]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import json
import drms
from astropy.time import Time
from pathlib import Path
import sunpy.map
import matplotlib.pyplot as plt
import os
import numpy as np
from astropy.io import fits
from PIL import Image
import pandas as pd

In [3]:
import json

def solarflare_query_processer(infile, output_info=1):
    # opening the file
    f = open(infile, 'r')
    # reading json part
    for line in f:
        if (line[0] == '{'):
            solarflare_obj = json.loads(line)
            break
    f.close()
    
    if (output_info == 1):
        print( "----------------------------------")
        print( "Possible dictionary keys (fields):")
        print ("----------------------------------")
        for key in solarflare_obj.keys():
            print(key)
        print ("---------------------------------------")
        print ("Example: use solarflare_obj['SOLID'][2] to access SOLID for the third flare")
    
    # returning headers and records
    return solarflare_obj

In [21]:
import drms
from astropy.time import Time
from pathlib import Path

JSOC_EMAIL = "652024260007@smail.nju.edu.cn"
HMI_SERIES = "hmi.Ic_45s"
OUT_DIR = Path("HMI")
OUT_DIR.mkdir(exist_ok=True)

client = drms.Client()
# 读取太阳耀斑数据文件
data = solarflare_query_processer('solarflareoutput_2026-05-12T00_53_31_45.txt', output_info=1)

# 访问第三个耀斑的SOLID标识符
#third_flare_id = data['SOLID'][2]

----------------------------------
Possible dictionary keys (fields):
----------------------------------
SOLID
UniqueID
Start Time
Peak Time
End Time
AR number
X, arcsec
Y, arcsec
GOES Class
GOES Max Temp, MK
GOES Max EM, 10<sup>49</sup> cm<sup>-3</sup>
GOES Duration, s
GOES T-EM Delay, s
RHESSI energy, keV
RHESSI duration, s
RHESSI peak, counts
HEK peak flux
HEK channel, <span>&#8491;</span>
---------------------------------------
Example: use solarflare_obj['SOLID'][2] to access SOLID for the third flare


In [23]:
# 找出 X, arcsec 或 Y, arcsec 为空的行索引
row_to_remove = []
for i in range(len(data['X, arcsec'])):
    if not data['X, arcsec'][i] or not data['Y, arcsec'][i]:
        row_to_remove.append(i)

print(f'去掉了 {len(row_to_remove)} 行')

# 过滤所有字段
data_remain = {
    key: [v for i, v in enumerate(values) if i not in row_to_remove]
    for key, values in data.items()
}

print(f'还剩 {len(data_remain["X, arcsec"])} 行')
'''
#展开了就是上面那个
data_remain = {}                          # 新建空字典
for key, values in data.items():          # 外层：遍历每个字段，如 'X, arcsec'、'GOES Class' 等
    filtered_values = []                  # 该字段过滤后的值列表
    for i, v in enumerate(values):        # 内层：遍历该字段的每一行，i 是行号，v 是值
        if i not in row_to_remove:        # 这行不在删除名单里
            filtered_values.append(v)     # 就保留
    data_remain[key] = filtered_values    # 把过滤后的列表存入新字典
'''


去掉了 0 行
还剩 3110 行


"\n#展开了就是上面那个\ndata_remain = {}                          # 新建空字典\nfor key, values in data.items():          # 外层：遍历每个字段，如 'X, arcsec'、'GOES Class' 等\n    filtered_values = []                  # 该字段过滤后的值列表\n    for i, v in enumerate(values):        # 内层：遍历该字段的每一行，i 是行号，v 是值\n        if i not in row_to_remove:        # 这行不在删除名单里\n            filtered_values.append(v)     # 就保留\n    data_remain[key] = filtered_values    # 把过滤后的列表存入新字典\n"

In [20]:
dataa={
    key: i for i in range(10) for key in ['a', 'b', 'c']
}
print(dataa)

{'a': 9, 'b': 9, 'c': 9}


In [24]:
# 按GOES Class首字母分成A/B/C/M/X五类
goes_classes = ['A', 'B', 'C', 'M', 'X']
data_by_class = {cls: {key: [] for key in data_remain.keys()} for cls in goes_classes}

for i in range(len(data_remain['GOES Class'])):
    goes_val = data_remain['GOES Class'][i]
    if goes_val and goes_val[0] in goes_classes:
        cls = goes_val[0]
        for key in data_remain.keys():
            data_by_class[cls][key].append(data_remain[key][i])

# 查看各类别数量
for cls in goes_classes:
    print(f"{cls}级耀斑: {len(data_by_class[cls]['GOES Class'])}个")

A级耀斑: 0个
B级耀斑: 930个
C级耀斑: 1999个
M级耀斑: 168个
X级耀斑: 13个


In [25]:
#保存时去掉ABCMX方便排序筛选
for key, value in data_by_class.items():
    for i in range(len(data_by_class[key]['GOES Class'])):
        data_by_class[key]['GOES Class'][i]=float(data_by_class[key]['GOES Class'][i][1:])

In [ ]:
#保存到本地excel中
for cls, cls_data in data_by_class.items():
    df = pd.DataFrame(cls_data)
    df.to_excel(f'{cls}级耀斑.xlsx', index=False)
    print(f'{cls}级耀斑: {len(df)}行 已保存')

A级耀斑: 0行 已保存
B级耀斑: 930行 已保存
C级耀斑: 1999行 已保存
M级耀斑: 168行 已保存
X级耀斑: 13行 已保存


In [ ]:
#检查发现TAI和UTC两个时间其实是不一样的
from astropy.time import Time
#TAI是原子钟时间，比UTC要多35秒的闰秒差距
def utc_to_tai_time(t_utc):
    return Time(t_utc, scale="utc").tai

t1=utc_to_tai_time(data_unique['Start Time'][0:3])
print(data_unique['Start Time'][0:3])
print(t1)

['2013-01-11 19:37:00', '2013-01-12 21:57:00', '2013-01-14 15:38:00']
['2013-01-11 19:37:35.000' '2013-01-12 21:57:35.000'
 '2013-01-14 15:38:35.000']
